# Extraction, Nettoyage et Génération des données à partir des données existantes

Dans ce document, il sera question de gérer les données reçues pour les transformer dans un unique DataFrame final exploitable. Pour ce faire, nous allons dans un premier temps extraire toutes les données existantes, les nettoyer puis les envoyer dans un unique fichier csv processed. Ce fichier, une fois chargé, représentera notre DataFrame.

Le plan d'action est donc le suivant :
- Importation des Librairies
- Extraction des Données
- Nettoyage des Données
- Génération du DataFrame

## 1. Importation des Librairies

In [ ]:
import os
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print('Les librairies ont été installées correctement !')

Les librairies ont été installées correctement !


## 2. Extraction des Données

Il y a plusieurs fichiers csv et un fichier tsv, on va d'abord charger chaque fichier séparément dans son propre DataFrame associé.

In [ ]:
data_folder_path = 'data/raw/'

df_grand_est = pd.read_csv(os.path.join(data_folder_path, 'donnees_grand_est.csv'), header=None,
    names=['ca_mensuel_k', 'date_fermeture', 'date_ouverture', 'nb_avis', 'note_moyenne', 'prix_moyen', 'nb_pizzas_differentes', 'type_etablissement', 'latitude', 'longitude'])
df_idf = pd.read_csv(os.path.join(data_folder_path, 'donnees_idf.csv'))

# Le fichier nord-ouest commence à partir de la ligne 2, il faut donc sauter la première ligne
df_nord_ouest = pd.read_csv(os.path.join(data_folder_path, 'donnees_nord_ouest.csv'), sep=',', skiprows=2)

# Le fichier sud-est est au format TSV, il faut donc spécifier le séparateur
df_sud_est = pd.read_csv(os.path.join(data_folder_path, 'donnees_sud_est.tsv'), sep='\t')

# Pour l'export de toutes les régions, il y a des séparateurs du style === NOM RÉGION ===, il faut donc les ignorer
df_toutes_regions = pd.read_csv(os.path.join(data_folder_path, 'export_toutes_regions.csv'), on_bad_lines='skip')

print('Les données ont été chargées correctement !')

print()
print('Voici un aperçu des données du Grand Est :')
print(df_grand_est.head())

print()
print('Voici un aperçu des données de l\'Île-de-France :')
print(df_idf.head())

print()
print('Voici un aperçu des données du Nord-Ouest :')
print(df_nord_ouest.head())

print()
print('Voici un aperçu des données du Sud-Est :')
print(df_sud_est.head())

print()
print('Voici un aperçu des données de toutes les régions :')
print(df_toutes_regions.head())

Les données ont été chargées correctement !

Voici un aperçu des données du Grand Est :
   ca_mensuel_k  date_fermeture  date_ouverture  nb_avis  note_moyenne  \
0           105            1999            1980     1800           3.0   
1            88            2009            1020     2117           0.0   
2            47            2006            1980      554           4.2   
3            20            1985             900      671           4.1   
4            54            2015            1920      654           4.9   

   prix_moyen  nb_pizzas_differentes  type_etablissement  latitude  longitude  
0       16.30                     37               SNACK   48.0824     6.9553  
1       14.55                     22               SNACK   49.8105     5.6683  
2       16.46                     15              chaine   48.3068     6.8964  
3        8.54                     33  restaurant ITALIEN   48.3035     4.3407  
4       15.63                     37              chaine   49.3249 

On va également afficher la structure de chaque fichier

In [ ]:
print('Voici un aperçu de la structure du Grand Est :')
print(df_grand_est.info())
print(f'Il y a {df_grand_est.shape[0]} lignes et {df_grand_est.shape[1]} colonnes dans le DataFrame du Grand Est.')

print()
print('Voici un aperçu de la structure de l\'Île-de-France :')
print(df_idf.info())
print(f'Il y a {df_idf.shape[0]} lignes et {df_idf.shape[1]} colonnes dans le DataFrame de l\'Île-de-France.')

print()
print('Voici un aperçu de la structure du Nord-Ouest :')
print(df_nord_ouest.info())
print(f'Il y a {df_nord_ouest.shape[0]} lignes et {df_nord_ouest.shape[1]} colonnes dans le DataFrame du Nord-Ouest.')

print()
print('Voici un aperçu de la structure du Sud-Est :')
print(df_sud_est.info())
print(f'Il y a {df_sud_est.shape[0]} lignes et {df_sud_est.shape[1]} colonnes dans le DataFrame du Sud-Est.')

print()
print('Voici un aperçu de la structure de toutes les régions :')
print(df_toutes_regions.info())
print(f'Il y a {df_toutes_regions.shape[0]} lignes et {df_toutes_regions.shape[1]} colonnes dans le DataFrame de toutes les régions.')

Voici un aperçu de la structure du Grand Est :
<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ca_mensuel_k           120 non-null    int64  
 1   date_fermeture         120 non-null    int64  
 2   date_ouverture         120 non-null    int64  
 3   nb_avis                120 non-null    int64  
 4   note_moyenne           120 non-null    float64
 5   prix_moyen             120 non-null    float64
 6   nb_pizzas_differentes  120 non-null    int64  
 7   type_etablissement     120 non-null    str    
 8   latitude               120 non-null    float64
 9   longitude              120 non-null    float64
dtypes: float64(4), int64(5), str(1)
memory usage: 9.5 KB
None
Il y a 120 lignes et 10 colonnes dans le DataFrame du Grand Est.

Voici un aperçu de la structure de l'Île-de-France :
<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0

## 3. Nettoyage des Données

On va maintenant passer au nettoyage des données. Dans un premier temps, on va normaliser le nom des colonnes.

In [ ]:
col_mapping = {
    'lat': 'latitude',
    'lon': 'longitude',
    'type': 'type_etablissement',
    'nb_pizzas': 'nb_pizzas_differentes',
    'prix': 'prix_moyen',
    'avis': 'nb_avis',
    'prep_time': 'temps_preparation_moyen',
    'opened': 'date_ouverture',
    'ca': 'ca_mensuel_k',

    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'Type': 'type_etablissement',
    'nb_pizzas_carte': 'nb_pizzas_differentes',
    'Prix': 'prix_moyen',
    'Avis': 'nb_avis',
    'annee_ouverture': 'date_ouverture',

    'establishment_type': 'type_etablissement',
    'avg_price': 'prix_moyen',
    'nb_reviews': 'nb_avis',
    'avg_rating': 'note_moyenne',
    'prep_time_min': 'temps_preparation_moyen',
    'opening_date': 'date_ouverture',
    'monthly_revenue_k': 'ca_mensuel_k',
}

# On renomme les lignes
for df in [df_grand_est, df_idf, df_nord_ouest, df_sud_est, df_toutes_regions]:
    df.rename(columns=col_mapping, inplace=True)

print('Les colonnes ont été renommées pour plus de cohérence !')

Les colonnes ont été renommées pour plus de cohérence !


On va maintenant fusionner ces dataframes dans un unique DataFrame pour pouvoir mieux l'exploiter.

In [ ]:
# On fusionne toutes les colonnes
df = pd.concat([df_grand_est, df_idf, df_nord_ouest, df_sud_est, df_toutes_regions], ignore_index=True)

# On normalise les colonnes dont on a besoin
df = df[['latitude', 'longitude', 'type_etablissement', 'nb_pizzas_differentes', 'prix_moyen', 'nb_avis', 'note_moyenne', 'date_ouverture', 'date_fermeture', 'temps_preparation_moyen', 'ca_mensuel_k']]

# On essaie de convertir les lignes numériques en lignes numériques
for col in df.columns:
    converted = pd.to_numeric(df[col], errors='coerce')
    # On ne remplace que si au moins une valeur a pu être convertie
    if converted.notna().sum() > 0:
        df[col] = converted

print('Les données ont été fusionnées en un seul DataFrame !')
print(f'Il y a {df.shape[0]} lignes et {df.shape[1]} colonnes dans le DataFrame fusionné.')

print()
print('Voici un aperçu du DataFrame fusionné :')
print(df.head())

print()
print('Voici un aperçu de la structure du DataFrame fusionné :')
print(df.info())

Les données ont été fusionnées en un seul DataFrame !
Il y a 564 lignes et 11 colonnes dans le DataFrame fusionné.

Voici un aperçu du DataFrame fusionné :
   latitude  longitude  type_etablissement  nb_pizzas_differentes  prix_moyen  \
0   48.0824     6.9553               SNACK                   37.0       16.30   
1   49.8105     5.6683               SNACK                   22.0       14.55   
2   48.3068     6.8964              chaine                   15.0       16.46   
3   48.3035     4.3407  restaurant ITALIEN                   33.0        8.54   
4   49.3249     6.4801              chaine                   37.0       15.63   

   nb_avis  note_moyenne  date_ouverture  date_fermeture  \
0   1800.0           3.0          1980.0          1999.0   
1   2117.0           0.0          1020.0          2009.0   
2    554.0           4.2          1980.0          2006.0   
3    671.0           4.1           900.0          1985.0   
4    654.0           4.9          1920.0          2015.0 

On commence par remarquer le nombre de valeurs nulles dans le DataFrame.

In [ ]:
print(f'Voici la liste des valeurs nulles :\n{df.isnull().sum()}')

Voici la liste des valeurs nulles :
latitude                   194
longitude                  194
type_etablissement           5
nb_pizzas_differentes        8
prix_moyen                 125
nb_avis                      5
note_moyenne               383
date_ouverture             135
date_fermeture             444
temps_preparation_moyen    161
ca_mensuel_k                 5
dtype: int64


On va donc remplacer les valeurs de nombres par leur valeur médiane et les str / object par leur mode.

In [ ]:
for col in df.columns:
    if df[col].dtype == 'int64' or df[col].dtype == 'float64':
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

print('Les valeurs nulles ont été traitées !')

Les valeurs nulles ont été traitées !


## 4. Génération du DataFrame

Enfin, nous pouvons générer le fichier CSV contenant les données aggrégées et nettoyées.

In [ ]:
final_folder_path = './data/processed/'
final_file_path = os.path.join(final_folder_path, 'final.csv')

df.to_csv(final_file_path, index=False)
print(f'Le fichier du DataFrame final a été sauvegardé ici : {final_file_path}')

Le fichier du DataFrame final a été sauvegardé ici : ./data/processed/final.csv
